[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/073.%20The%20Vehicle%20Routing%20Problem%20with%20Time%20Windows%20%28VRPTW%29/P73-Tier-7_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# Problem 73: The Vehicle Routing Problem with Time Windows (VRPTW)

## Tier 7: Human–AI Partnership

### Purpose and scope

This notebook presents an illustrative human–AI collaboration workflow for decisions related to logistics. It models how an AI recommendation can be explained, reviewed by a domain expert, modified, and evaluated. The examples include route optimization, emergency response, ethical prioritization, and contextual adjustment.

The notebook does **not** formulate or solve a vehicle routing problem with time windows. It contains no customer coordinates, travel-time matrix, vehicle-capacity constraints, or time-window constraints. Its recommendations and performance scores are simulated examples of a human–AI decision process, not outputs from a routing optimizer or measured operational results.

### Modeling assumptions

- A recommendation is represented by a solution dictionary, a confidence score, explanatory text, possible alternatives, risk factors, and predicted outcomes.
- Confidence, human review, feedback, and outcome quality are generated by illustrative rules and seeded random draws. They are not calibrated from real data.
- The expert review process is simulated. It does not represent a real user interface, human-subject study, or validated emergency protocol.
- Baseline performance scores used for comparison are illustrative reference values. The resulting percentage comparison describes scores, not delivery speed or causal improvement.
- Wall-clock timings depend on the machine and runtime environment; they are not reproducible performance benchmarks.

### Workflow

For each scenario, the system generates a recommendation, confidence score, reasoning, alternatives, risks, and expected outcomes. A selected expert reviews the recommendation. The collaboration step combines the proposed solution with any simulated human modifications, records the interaction, and updates the expert's trust and the system's review threshold. Finally, the notebook summarizes the simulated collaborations and visualizes selected measures.

### How to interpret the results

The outputs demonstrate the structure of an explainable, human-in-the-loop workflow. A high simulated confidence score is not proof that a recommendation is correct, and an outcome-quality score is not an observed service outcome. In real logistics operations, route feasibility, time windows, safety, equity, and operational performance would need to be checked against relevant data and explicit constraints.

In [ ]:
import random
import time
from dataclasses import dataclass
from enum import Enum
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# A fixed seed makes the simulated random draws reproducible in a fresh run.
np.random.seed(42)
random.seed(42)

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
class DecisionType(Enum):
    ROUTE_OPTIMIZATION = "route_optimization"
    EMERGENCY_RESPONSE = "emergency_response"
    ETHICAL_PRIORITIZATION = "ethical_prioritization"
    CONTEXTUAL_ADJUSTMENT = "contextual_adjustment"


@dataclass
class AIRecommendation:
    """A simulated recommendation and its explanatory information."""

    recommendation_id: str
    decision_type: DecisionType
    proposed_solution: Dict[str, Any]
    confidence_score: float
    reasoning: List[str]
    alternatives: List[Dict[str, Any]]
    risk_factors: List[str]
    expected_outcomes: Dict[str, float]
    requires_human_review: bool
    timestamp: float


@dataclass
class HumanExpert:
    """A simulated domain expert participating in the workflow."""

    id: str
    name: str
    expertise_areas: List[str]
    experience_years: int
    trust_level: float
    decision_history: List[Dict[str, Any]]
    feedback_patterns: Dict[str, Any]


@dataclass
class HumanAICollaboration:
    """A record of one simulated human–AI decision interaction."""

    collaboration_id: str
    ai_recommendation: AIRecommendation
    human_input: Dict[str, Any]
    final_decision: Dict[str, Any]
    human_confidence: float
    ai_confidence: float
    collaboration_time: float
    outcome_quality: float
    learning_insights: List[str]


@dataclass
class ExplainableAI:
    """Configuration and records for the illustrative AI component."""

    model_version: str
    confidence_threshold: float
    explanation_depth: int
    learning_rate: float
    human_feedback_weight: float
    trust_metrics: Dict[str, Any]
    performance_history: List[Dict[str, Any]]

In [ ]:
class HumanAIPartnership:
    """Simulate explainable recommendations and expert review."""

    def __init__(self) -> None:
        self.ai_system = ExplainableAI(
            model_version="1.0",
            confidence_threshold=0.85,
            explanation_depth=3,
            learning_rate=0.1,
            human_feedback_weight=0.3,
            trust_metrics={},
            performance_history=[],
        )
        self.human_experts: Dict[str, HumanExpert] = {}
        self.collaboration_history: List[HumanAICollaboration] = []
        self.current_time = 0.0
        self.total_collaborations = 0

        # These are illustrative score baselines, not measured outcomes.
        self.ai_only_performance = 0.65
        self.human_only_performance = 0.72
        self.collaborative_performance = 0.0
        self.trust_score = 0.0

    def add_human_expert(self, expert: HumanExpert) -> None:
        """Register an expert so scenarios can select them by ID."""
        self.human_experts[expert.id] = expert

    def generate_ai_recommendation(
        self,
        problem_context: Dict[str, Any],
        decision_type: DecisionType,
    ) -> AIRecommendation:
        """Build a simulated recommendation and its supporting information."""
        confidence = self._calculate_confidence(problem_context, decision_type)
        proposed_solution = self._generate_solution(problem_context, decision_type)
        reasoning = self._generate_reasoning(
            proposed_solution, problem_context, decision_type
        )
        alternatives = self._generate_alternatives(proposed_solution)
        risk_factors = self._identify_risk_factors(
            proposed_solution, problem_context
        )
        expected_outcomes = self._predict_outcomes(proposed_solution)
        requires_human_review = (
            confidence < self.ai_system.confidence_threshold
            or decision_type
            in [
                DecisionType.EMERGENCY_RESPONSE,
                DecisionType.ETHICAL_PRIORITIZATION,
            ]
        )

        return AIRecommendation(
            recommendation_id=(
                f"AI_{self.current_time}_"
                f"{random.randint(1000, 9999)}"
            ),
            decision_type=decision_type,
            proposed_solution=proposed_solution,
            confidence_score=confidence,
            reasoning=reasoning,
            alternatives=alternatives,
            risk_factors=risk_factors,
            expected_outcomes=expected_outcomes,
            requires_human_review=requires_human_review,
            timestamp=self.current_time,
        )

    def _calculate_confidence(
        self, context: Dict[str, Any], decision_type: DecisionType
    ) -> float:
        """Assign a simulated confidence score based on type and complexity."""
        base_confidence = {
            DecisionType.ROUTE_OPTIMIZATION: 0.92,
            DecisionType.EMERGENCY_RESPONSE: 0.75,
            DecisionType.ETHICAL_PRIORITIZATION: 0.68,
            DecisionType.CONTEXTUAL_ADJUSTMENT: 0.80,
        }[decision_type]

        complexity_factor = float(context.get("complexity", 1.0))
        confidence = base_confidence / (1 + complexity_factor * 0.1)
        confidence += np.random.normal(0, 0.05)
        return float(np.clip(confidence, 0.0, 1.0))

    def _generate_solution(
        self, context: Dict[str, Any], decision_type: DecisionType
    ) -> Dict[str, Any]:
        """Return a fixed illustrative solution for the selected decision type."""
        if decision_type == DecisionType.ROUTE_OPTIMIZATION:
            return {
                "routes": [[1, 3, 5], [2, 4, 6]],
                "total_distance": 145.2,
                "time_window_compliance": 0.94,
                "vehicle_utilization": 0.87,
            }
        if decision_type == DecisionType.EMERGENCY_RESPONSE:
            return {
                "priority_routes": [[1, 2], [3, 4], [5, 6]],
                "emergency_supplies": ["medical", "food", "water"],
                "response_time": 12.5,
                "coverage_area": 0.89,
            }
        if decision_type == DecisionType.ETHICAL_PRIORITIZATION:
            return {
                "vulnerable_customers": [1, 3, 5],
                "priority_routes": [[1, 5], [3], [2, 4, 6]],
                "equity_score": 0.91,
                "service_fairness": 0.88,
            }
        return {
            "adaptive_routes": [[1, 4], [2, 5, 6], [3]],
            "flexibility_score": 0.85,
            "adaptation_time": 8.3,
        }

    def _generate_reasoning(
        self,
        solution: Dict[str, Any],
        context: Dict[str, Any],
        decision_type: DecisionType,
    ) -> List[str]:
        """Describe the illustrative factors associated with a recommendation."""
        if decision_type == DecisionType.ROUTE_OPTIMIZATION:
            return [
                "The route proposal is represented as two ordered customer sequences.",
                (
                    "The illustrative time-window compliance score is "
                    f"{solution['time_window_compliance']:.1%}."
                ),
                (
                    "The illustrative vehicle-utilization score is "
                    f"{solution['vehicle_utilization']:.1%}."
                ),
                "No travel-time or time-window feasibility calculation is performed.",
            ]
        if decision_type == DecisionType.EMERGENCY_RESPONSE:
            return [
                "The example groups affected locations into priority routes.",
                (
                    "The illustrative response-time value is "
                    f"{solution['response_time']:.1f} minutes."
                ),
                (
                    "The illustrative coverage score is "
                    f"{solution['coverage_area']:.1%}."
                ),
                "A human expert is required to review emergency-response decisions.",
            ]
        if decision_type == DecisionType.ETHICAL_PRIORITIZATION:
            return [
                "The example marks a subset of customers for priority consideration.",
                (
                    "The illustrative equity score is "
                    f"{solution['equity_score']:.2f}."
                ),
                (
                    "The illustrative service-fairness score is "
                    f"{solution['service_fairness']:.1%}."
                ),
                "These scores do not establish fairness without a defined measure and data.",
            ]
        return [
            "The example assigns customer sequences to adaptive routes.",
            (
                "The illustrative flexibility score is "
                f"{solution['flexibility_score']:.2f}."
            ),
            (
                "The illustrative adaptation time is "
                f"{solution['adaptation_time']:.1f} minutes."
            ),
            "The routes are example data, not an operational routing plan.",
        ]

    def _generate_alternatives(
        self, primary_solution: Dict[str, Any]
    ) -> List[Dict[str, Any]]:
        """Create two simple illustrative variants of the primary solution."""
        alternatives: List[Dict[str, Any]] = []

        conservative = primary_solution.copy()
        if "total_distance" in conservative:
            conservative["total_distance"] *= 1.1
        if "time_window_compliance" in conservative:
            conservative["time_window_compliance"] *= 0.95
        alternatives.append(conservative)

        aggressive = primary_solution.copy()
        if "total_distance" in aggressive:
            aggressive["total_distance"] *= 0.9
        if "response_time" in aggressive:
            aggressive["response_time"] *= 1.15
        alternatives.append(aggressive)
        return alternatives

    def _identify_risk_factors(
        self, solution: Dict[str, Any], context: Dict[str, Any]
    ) -> List[str]:
        """Flag risks using the example context and solution values."""
        risks: List[str] = []

        if context.get("weather_conditions", "clear") != "clear":
            risks.append("Adverse weather may increase travel times.")
        if context.get("traffic_level", "normal") in ["high", "congested"]:
            risks.append("Heavy traffic may cause delays.")
        if solution.get("total_distance", 0) > 150:
            risks.append("The example's high total distance may increase fuel use.")
        if solution.get("response_time", 0) > 15:
            risks.append("The example's response time may be unsuitable for urgent needs.")
        return risks

    def _predict_outcomes(
        self, solution: Dict[str, Any]
    ) -> Dict[str, float]:
        """Generate illustrative, noisy outcome scores."""
        outcomes = {
            "success_probability": float(0.85 + np.random.normal(0, 0.1)),
            "cost_efficiency": float(0.78 + np.random.normal(0, 0.08)),
            "customer_satisfaction": float(0.82 + np.random.normal(0, 0.07)),
        }

        if "time_window_compliance" in solution:
            outcomes["customer_satisfaction"] *= solution["time_window_compliance"]
        if "equity_score" in solution:
            outcomes["social_acceptance"] = float(solution["equity_score"])
        return outcomes

    def human_ai_collaboration(
        self,
        ai_recommendation: AIRecommendation,
        human_expert: HumanExpert,
    ) -> HumanAICollaboration:
        """Review, combine, record, and learn from one recommendation."""
        collaboration_start = time.time()
        human_input = self._human_review_process(ai_recommendation, human_expert)
        final_decision = self._collaborative_decision(
            ai_recommendation, human_input
        )
        outcome_quality = self._simulate_outcome_quality(human_input)
        learning_insights = self._generate_learning_insights(
            ai_recommendation, human_input
        )
        collaboration_time = time.time() - collaboration_start

        collaboration = HumanAICollaboration(
            collaboration_id=(
                f"COL_{self.current_time}_"
                f"{random.randint(1000, 9999)}"
            ),
            ai_recommendation=ai_recommendation,
            human_input=human_input,
            final_decision=final_decision,
            human_confidence=human_input["confidence"],
            ai_confidence=ai_recommendation.confidence_score,
            collaboration_time=collaboration_time,
            outcome_quality=outcome_quality,
            learning_insights=learning_insights,
        )

        self._update_trust(human_expert, collaboration)
        self._update_ai_learning(collaboration)
        self.collaboration_history.append(collaboration)
        self.total_collaborations += 1
        human_expert.decision_history.append(
            {
                "recommendation_id": ai_recommendation.recommendation_id,
                "outcome_quality": outcome_quality,
            }
        )
        return collaboration

    def _human_review_process(
        self, ai_recommendation: AIRecommendation, expert: HumanExpert
    ) -> Dict[str, Any]:
        """Simulate an expert's assessment and possible modifications."""
        evaluation = str(
            np.random.choice(
                ["excellent", "good", "adequate", "needs_improvement"],
                p=[0.3, 0.4, 0.2, 0.1],
            )
        )
        human_input: Dict[str, Any] = {
            "review_timestamp": self.current_time,
            "expert_id": expert.id,
            "expertise_applied": random.choice(expert.expertise_areas),
            "ai_confidence_assessment": float(
                np.clip(
                    ai_recommendation.confidence_score
                    + np.random.normal(0, 0.1),
                    0.0,
                    1.0,
                )
            ),
            "reasoning_evaluation": evaluation,
            "risk_assessment": str(
                np.random.choice(["low", "medium", "high"], p=[0.4, 0.4, 0.2])
            ),
            "suggested_modifications": {},
        }

        if evaluation in ["needs_improvement", "adequate"]:
            human_input["suggested_modifications"] = {
                "route_adjustments": "Review priority locations with the expert.",
                "timing_changes": "Consider a buffer for time-sensitive supplies.",
                "resource_reallocation": "Review vehicle availability for urgent requests.",
            }

        confidence_adjustments = {
            "excellent": 0.1,
            "good": 0.0,
            "adequate": -0.1,
            "needs_improvement": -0.2,
        }
        human_input["confidence"] = float(
            np.clip(
                expert.trust_level + confidence_adjustments[evaluation],
                0.0,
                1.0,
            )
        )
        return human_input

    def _collaborative_decision(
        self,
        ai_rec: AIRecommendation,
        human_input: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Attach review suggestions and provenance to the AI proposal."""
        final_decision = ai_rec.proposed_solution.copy()
        modifications = human_input["suggested_modifications"]
        for key, value in modifications.items():
            final_decision[f"human_{key}"] = value

        final_decision["collaboration_metadata"] = {
            "ai_confidence": ai_rec.confidence_score,
            "human_confidence": human_input["confidence"],
            "expertise_applied": human_input["expertise_applied"],
            "decision_type": ai_rec.decision_type.value,
        }
        return final_decision

    def _simulate_outcome_quality(self, human_input: Dict[str, Any]) -> float:
        """Assign a simulated quality score from review category and noise."""
        base_quality = 0.75
        if human_input["reasoning_evaluation"] == "excellent":
            base_quality += 0.15
        elif human_input["reasoning_evaluation"] == "good":
            base_quality += 0.08
        if human_input["suggested_modifications"]:
            base_quality += 0.05
        base_quality += np.random.normal(0, 0.05)
        return float(np.clip(base_quality, 0.0, 1.0))

    def _generate_learning_insights(
        self,
        ai_rec: AIRecommendation,
        human_input: Dict[str, Any],
    ) -> List[str]:
        """Record feedback themes from the simulated review."""
        insights: List[str] = []
        if human_input["reasoning_evaluation"] != "excellent":
            insights.append(
                "Review the explanation for this decision type."
            )
        if human_input["suggested_modifications"]:
            insights.append(
                "The expert identified contextual factors for review."
            )
            insights.append(
                "Record the domain knowledge represented by the suggestions."
            )
        if ai_rec.confidence_score < 0.8:
            insights.append(
                "Review whether the confidence threshold is appropriate."
            )
        insights.append(
            f"Expertise applied: {human_input['expertise_applied']}."
        )
        return insights

    def _update_trust(
        self,
        expert: HumanExpert,
        collaboration: HumanAICollaboration,
    ) -> None:
        """Adjust simulated trust in response to the simulated quality score."""
        if collaboration.outcome_quality > 0.8:
            expert.trust_level += 0.02
        elif collaboration.outcome_quality < 0.6:
            expert.trust_level -= 0.01
        expert.trust_level = float(np.clip(expert.trust_level, 0.0, 1.0))

    def _update_ai_learning(
        self, collaboration: HumanAICollaboration
    ) -> None:
        """Adjust the review threshold using the simulated expert evaluation."""
        evaluation = collaboration.human_input["reasoning_evaluation"]
        if evaluation == "excellent":
            self.ai_system.confidence_threshold *= 0.98
        elif evaluation == "needs_improvement":
            self.ai_system.confidence_threshold *= 1.02
        self.ai_system.confidence_threshold = float(
            np.clip(self.ai_system.confidence_threshold, 0.7, 0.95)
        )

    def calculate_partnership_metrics(self) -> Dict[str, float]:
        """Summarize recorded collaborations and the illustrative baseline."""
        if not self.collaboration_history:
            return {
                "avg_outcome_quality": 0.0,
                "avg_collaboration_time": 0.0,
                "trust_score": 0.0,
                "learning_effectiveness": 0.0,
                "collaborative_improvement": 0.0,
            }

        avg_outcome_quality = float(
            np.mean([c.outcome_quality for c in self.collaboration_history])
        )
        avg_collaboration_time = float(
            np.mean([c.collaboration_time for c in self.collaboration_history])
        )
        if self.human_experts:
            self.trust_score = float(
                np.mean([expert.trust_level for expert in self.human_experts.values()])
            )

        insight_count = sum(
            len(collaboration.learning_insights)
            for collaboration in self.collaboration_history
        )
        learning_effectiveness = insight_count / len(self.collaboration_history)
        self.collaborative_performance = avg_outcome_quality
        best_individual_baseline = max(
            self.ai_only_performance, self.human_only_performance
        )
        collaborative_improvement = (
            (self.collaborative_performance - best_individual_baseline)
            / best_individual_baseline
            * 100
        )

        return {
            "avg_outcome_quality": avg_outcome_quality,
            "avg_collaboration_time": avg_collaboration_time,
            "trust_score": self.trust_score,
            "learning_effectiveness": float(learning_effectiveness),
            "collaborative_improvement": float(collaborative_improvement),
        }


partnership = HumanAIPartnership()
print("Human–AI partnership simulation initialized.")

In [ ]:
def create_human_experts(
    partnership: HumanAIPartnership,
) -> List[HumanExpert]:
    """Create and register the illustrative domain experts."""
    experts = [
        HumanExpert(
            id="expert_001",
            name="Dr. Sarah Chen",
            expertise_areas=[
                "emergency_response",
                "medical_logistics",
                "disaster_management",
            ],
            experience_years=15,
            trust_level=0.85,
            decision_history=[],
            feedback_patterns={},
        ),
        HumanExpert(
            id="expert_002",
            name="Michael Rodriguez",
            expertise_areas=[
                "route_optimization",
                "traffic_management",
                "urban_planning",
            ],
            experience_years=12,
            trust_level=0.78,
            decision_history=[],
            feedback_patterns={},
        ),
        HumanExpert(
            id="expert_003",
            name="Dr. Aisha Patel",
            expertise_areas=[
                "ethics",
                "social_justice",
                "community_planning",
                "vulnerable_populations",
            ],
            experience_years=18,
            trust_level=0.92,
            decision_history=[],
            feedback_patterns={},
        ),
        HumanExpert(
            id="expert_004",
            name="James Thompson",
            expertise_areas=[
                "operations_management",
                "logistics_coordination",
                "resource_allocation",
            ],
            experience_years=10,
            trust_level=0.73,
            decision_history=[],
            feedback_patterns={},
        ),
    ]

    for expert in experts:
        partnership.add_human_expert(expert)

    print(f"Registered {len(experts)} illustrative human experts:")
    for expert in experts:
        expertise = ", ".join(expert.expertise_areas)
        print(
            f"  {expert.name}: {expertise} "
            f"({expert.experience_years} years of stated experience)"
        )
    return experts


human_experts = create_human_experts(partnership)

In [ ]:
def simulate_emergency_response_scenario(
    partnership: HumanAIPartnership,
) -> Dict[str, Any]:
    """Run one illustrative flood-response collaboration scenario."""
    print("\n" + "=" * 60)
    print("EMERGENCY RESPONSE SCENARIO: FLOOD")
    print("=" * 60)

    problem_context = {
        "disaster_type": "flood",
        "affected_areas": 15,
        "urgency_level": "critical",
        "available_vehicles": 6,
        "emergency_supplies": ["medical", "food", "water", "blankets"],
        "weather_conditions": "rain",
        "traffic_level": "congested",
        "complexity": 2.5,
    }

    print("\nIllustrative scenario context:")
    print(f"  Disaster: {problem_context['disaster_type'].title()}")
    print(f"  Affected areas: {problem_context['affected_areas']}")
    print(f"  Available vehicles: {problem_context['available_vehicles']}")
    print(
        "  Supplies: "
        + ", ".join(problem_context["emergency_supplies"])
    )

    print("\nGenerating the simulated AI recommendation...")
    ai_start = time.time()
    ai_recommendation = partnership.generate_ai_recommendation(
        problem_context, DecisionType.EMERGENCY_RESPONSE
    )
    ai_time = time.time() - ai_start

    print(f"Recommendation generation time: {ai_time:.4f} seconds")
    print(f"Simulated AI confidence: {ai_recommendation.confidence_score:.2f}")
    print(f"Human review required: {ai_recommendation.requires_human_review}")
    print("\nRecommendation reasoning:")
    for number, reason in enumerate(ai_recommendation.reasoning, start=1):
        print(f"  {number}. {reason}")

    selected_expert = partnership.human_experts["expert_001"]
    print(f"\nSelected expert: {selected_expert.name}")
    print(f"Expertise areas: {', '.join(selected_expert.expertise_areas)}")

    print("\nRunning the simulated human review...")
    collaboration = partnership.human_ai_collaboration(
        ai_recommendation, selected_expert
    )
    print(
        "Collaboration processing time: "
        f"{collaboration.collaboration_time:.4f} seconds"
    )
    print(f"Simulated human confidence: {collaboration.human_confidence:.2f}")
    print(f"Simulated AI confidence: {collaboration.ai_confidence:.2f}")
    print(f"Simulated outcome-quality score: {collaboration.outcome_quality:.2f}")

    modifications = collaboration.human_input["suggested_modifications"]
    print("\nSimulated expert suggestions:")
    if modifications:
        for key, value in modifications.items():
            print(f"  {key}: {value}")
    else:
        print("  No modifications suggested in this simulated review.")

    print("\nRecorded learning insights:")
    for insight in collaboration.learning_insights:
        print(f"  - {insight}")

    best_individual_baseline = max(
        partnership.ai_only_performance,
        partnership.human_only_performance,
    )
    collaborative_improvement = (
        (collaboration.outcome_quality - best_individual_baseline)
        / best_individual_baseline
        * 100
    )
    print("\nIllustrative score comparison:")
    print(f"  AI-only reference score: {partnership.ai_only_performance:.2f}")
    print(f"  Human-only reference score: {partnership.human_only_performance:.2f}")
    print(f"  Collaborative simulated score: {collaboration.outcome_quality:.2f}")
    print(
        "  Difference from the higher reference score: "
        f"{collaborative_improvement:.1f}%"
    )
    print("  This is a score comparison, not a delivery-time comparison.")

    return {
        "problem_context": problem_context,
        "ai_recommendation": ai_recommendation,
        "collaboration": collaboration,
        "ai_time": ai_time,
        "collaboration_time": collaboration.collaboration_time,
        "ai_only_performance": partnership.ai_only_performance,
        "human_only_performance": partnership.human_only_performance,
        "collaborative_improvement": collaborative_improvement,
    }


emergency_results = simulate_emergency_response_scenario(partnership)

In [ ]:
def simulate_multiple_collaborations(
    partnership: HumanAIPartnership,
) -> Dict[str, List[Dict[str, Any]]]:
    """Run illustrative collaborations for three decision types."""
    print("\n" + "=" * 60)
    print("MULTIPLE COLLABORATION SCENARIOS")
    print("=" * 60)

    scenarios = [
        {
            "name": "Route Optimization",
            "decision_type": DecisionType.ROUTE_OPTIMIZATION,
            "expert_id": "expert_002",
            "context": {
                "customers": 12,
                "vehicles": 4,
                "complexity": 1.2,
                "traffic_level": "moderate",
            },
        },
        {
            "name": "Ethical Prioritization",
            "decision_type": DecisionType.ETHICAL_PRIORITIZATION,
            "expert_id": "expert_003",
            "context": {
                "vulnerable_customers": 8,
                "equity_requirements": "high",
                "complexity": 2.0,
                "social_factors": ["elderly", "low_income", "disabled"],
            },
        },
        {
            "name": "Contextual Adjustment",
            "decision_type": DecisionType.CONTEXTUAL_ADJUSTMENT,
            "expert_id": "expert_004",
            "context": {
                "dynamic_demand": True,
                "resource_constraints": "medium",
                "complexity": 1.5,
                "flexibility_required": 0.3,
            },
        },
    ]

    results: List[Dict[str, Any]] = []
    for scenario in scenarios:
        expert = partnership.human_experts[scenario["expert_id"]]
        print(f"\nScenario: {scenario['name']}")
        print(f"Selected expert: {expert.name}")

        ai_recommendation = partnership.generate_ai_recommendation(
            scenario["context"], scenario["decision_type"]
        )
        collaboration = partnership.human_ai_collaboration(
            ai_recommendation, expert
        )
        results.append(
            {
                "scenario_name": scenario["name"],
                "decision_type": scenario["decision_type"].value,
                "expert_name": expert.name,
                "ai_confidence": ai_recommendation.confidence_score,
                "human_confidence": collaboration.human_confidence,
                "outcome_quality": collaboration.outcome_quality,
                "collaboration_time": collaboration.collaboration_time,
                "learning_insights": len(collaboration.learning_insights),
            }
        )
        print(f"  Simulated AI confidence: {ai_recommendation.confidence_score:.2f}")
        print(f"  Simulated human confidence: {collaboration.human_confidence:.2f}")
        print(f"  Simulated outcome-quality score: {collaboration.outcome_quality:.2f}")
        print(f"  Recorded learning insights: {len(collaboration.learning_insights)}")

    return {"scenarios": results}


multiple_results = simulate_multiple_collaborations(partnership)

In [ ]:
def visualize_human_ai_partnership(
    emergency: Dict[str, Any],
    multiple: Dict[str, List[Dict[str, Any]]],
    partnership: HumanAIPartnership,
) -> None:
    """Plot measured simulation values and summarize their meaning."""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2, figsize=(16, 12)
    )

    # Only the two recorded elapsed times are shown; review sub-stages are not timed.
    time_labels = ["AI recommendation", "Human–AI collaboration"]
    time_values = [emergency["ai_time"], emergency["collaboration_time"]]
    bars = ax1.bar(
        time_labels,
        time_values,
        color=["steelblue", "darkorange"],
        alpha=0.8,
    )
    ax1.set_ylabel("Elapsed time (seconds)")
    ax1.set_title("Observed Runtime of Simulation Steps")
    ax1.tick_params(axis="x", rotation=15)
    ax1.grid(axis="y", alpha=0.3)
    for bar, value in zip(bars, time_values):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.4f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    performance_labels = ["AI-only reference", "Human-only reference", "Collaboration"]
    performance_values = [
        emergency["ai_only_performance"],
        emergency["human_only_performance"],
        emergency["collaboration"].outcome_quality,
    ]
    bars = ax2.bar(
        performance_labels,
        performance_values,
        color=["indianred", "steelblue", "seagreen"],
        alpha=0.8,
    )
    ax2.set_ylabel("Illustrative score")
    ax2.set_title("Simulated Score and Reference Values")
    ax2.set_ylim(0, 1)
    ax2.tick_params(axis="x", rotation=15)
    ax2.grid(axis="y", alpha=0.3)
    for bar, value in zip(bars, performance_values):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.015,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    scenarios = multiple["scenarios"]
    scenario_names = [scenario["scenario_name"] for scenario in scenarios]
    ai_confidences = [scenario["ai_confidence"] for scenario in scenarios]
    human_confidences = [scenario["human_confidence"] for scenario in scenarios]
    outcome_qualities = [scenario["outcome_quality"] for scenario in scenarios]
    x_positions = np.arange(len(scenario_names))
    bar_width = 0.25

    ax3.bar(
        x_positions - bar_width,
        ai_confidences,
        bar_width,
        label="AI confidence",
        color="steelblue",
        alpha=0.8,
    )
    ax3.bar(
        x_positions,
        human_confidences,
        bar_width,
        label="Human confidence",
        color="seagreen",
        alpha=0.8,
    )
    ax3.bar(
        x_positions + bar_width,
        outcome_qualities,
        bar_width,
        label="Outcome-quality score",
        color="darkorange",
        alpha=0.8,
    )
    ax3.set_xlabel("Illustrative scenario")
    ax3.set_ylabel("Score")
    ax3.set_title("Simulated Confidence and Outcome Scores")
    ax3.set_xticks(x_positions)
    ax3.set_xticklabels(scenario_names, rotation=25, ha="right")
    ax3.set_ylim(0, 1)
    ax3.legend()
    ax3.grid(axis="y", alpha=0.3)

    ax4.axis("off")
    metrics = partnership.calculate_partnership_metrics()
    average_experience = np.mean(
        [expert.experience_years for expert in partnership.human_experts.values()]
    )
    summary = (
        "Human–AI Partnership Simulation\n"
        "================================\n\n"
        "Emergency scenario\n"
        f"AI recommendation runtime: {emergency['ai_time']:.4f} s\n"
        "Collaboration runtime: "
        f"{emergency['collaboration_time']:.4f} s\n"
        "Simulated outcome-quality score: "
        f"{emergency['collaboration'].outcome_quality:.3f}\n\n"
        "Partnership records\n"
        f"Registered experts: {len(partnership.human_experts)}\n"
        f"Stated average experience: {average_experience:.1f} years\n"
        f"Recorded collaborations: {partnership.total_collaborations}\n"
        f"Average simulated quality: {metrics['avg_outcome_quality']:.3f}\n"
        f"Average collaboration runtime: {metrics['avg_collaboration_time']:.4f} s\n"
        f"Average trust score: {metrics['trust_score']:.3f}\n"
        "Learning insights per collaboration: "
        f"{metrics['learning_effectiveness']:.2f}\n\n"
        "Interpretation\n"
        "Scores and recommendations are simulated. The reference scores "
        "are assumptions, and elapsed times depend on the runtime environment."
    )
    ax4.text(
        0.02,
        0.98,
        summary,
        transform=ax4.transAxes,
        fontsize=9,
        verticalalignment="top",
        fontfamily="monospace",
    )

    fig.suptitle("Illustrative Human–AI Collaboration Measures", fontsize=15)
    plt.tight_layout()
    plt.show()


visualize_human_ai_partnership(
    emergency_results,
    multiple_results,
    partnership,
)

## Summary and Conclusions

### What the simulation demonstrates

The workflow makes several parts of a human-in-the-loop decision process explicit: the recommendation, its confidence score, its reasoning, possible alternatives, identified risks, and predicted outcomes. It then records a simulated expert review, incorporates any suggestions into the decision record, and updates trust and the confidence threshold according to simple feedback rules.

The emergency-response example and the three additional decision scenarios use the same sequence. Their displayed scores are generated by the notebook's rules and seeded random draws. The AI-only and human-only values are illustrative reference scores. Accordingly, a reported percentage difference is a comparison between scores, not evidence that collaboration improves delivery speed, quality, or real-world outcomes.

### How to read the measures

- **Confidence scores** are simulated assessments. They are not calibrated probabilities of correctness.
- **Outcome-quality scores** are generated from review categories and random variation. They are not measured operational results.
- **Trust scores** change according to the simulated outcome-quality score. They do not measure the trust of real experts.
- **Learning effectiveness** is the number of recorded learning insights per collaboration. It measures the simulation's logged feedback volume, not model accuracy or learning progress.
- **Runtime measurements** are wall-clock timings for code execution in the current environment. They are not service-time or routing-performance estimates.

### Relationship to a VRPTW model

A vehicle routing problem with time windows requires routing decisions to be checked against a defined network, travel times, customer demands, vehicle resources, and service windows. The route lists in this notebook are illustrative values; this workflow does not compute their distances or verify their feasibility. A production decision-support system would need to connect human review to a separately specified and validated routing model, while making the model's constraints and any infeasibility visible to reviewers.

### Strengths and limitations of this tier

A human–AI workflow can make room for contextual knowledge, ethical judgment, and explicit review of high-stakes recommendations. It also introduces practical requirements: experts must be available, explanations must be understandable, feedback must be recorded responsibly, and decision quality must be evaluated using suitable evidence. Human review can add time and may introduce bias; the presence of a human reviewer does not by itself guarantee a safe or fair decision.

### Further study

Natural extensions include defining measurable review criteria, recording feedback with appropriate privacy safeguards, testing the workflow on validated operational data, and evaluating recommendations against explicit feasibility and service requirements. Such work would require a clearly specified routing model and empirical evidence; the present simulation does not implement those extensions.